# Library 02 – Extracting batting partnerships

This is the most important part of the project. If a partnership boundary is wrong, every edge weight after
it is wrong too. So the rules are kept simple and written down.

**What is a partnership?** The time in an innings when one pair of batters is at the crease together. The pair
has no order: Root–Brook is the same pair as Brook–Root.

**Rules used by the extractor**

1. Deliveries are read in the order of the file: innings → over → ball.
2. Each delivery adds `runs.total` **once** to the pair at the crease. `runs.total` already includes extras,
   so extras are never added a second time.
3. If a delivery has a wicket record for one of the two batters, the partnership closes **after** that ball.
   The runs of the wicket ball still belong to the old pair. We do not use a fixed list of dismissal types:
   any wicket record that names a batter at the crease ends the partnership.
4. If the pair changes **without** a wicket, we accept it only when the file records a batter replacement.
   Otherwise we log it as `unexplained_pair_change`. We never carry the old pair forward silently.
5. The open partnership is closed once at the end of the innings.
6. If the same two players separate and later bat together again, that is **two** observations.
7. Penalty runs given to the whole innings do not belong to any ball, so they are never given to a pair.

Needs: `lib_00_settings`, `lib_01_data_loading`.

In [ ]:
@dataclass
class PartnershipObservation:
    """One partnership: one pair of batters, one unbroken stay at the crease."""

    match_id: str
    innings_index: int            # position of the innings in the match file (1 = first innings)
    partnership_seq: int          # 1 for the first partnership of the innings, 2 for the second, ...
    player_1_id: str              # the two players are stored in ID order, so a pair has one fixed key
    player_1_name: str
    player_2_id: str
    player_2_name: str
    partnership_runs: int = 0     # sum of runs.total over the deliveries of this partnership
    batter_runs: int = 0          # runs off the bat (kept for checking)
    extras_runs: int = 0          # extras (kept for checking)
    deliveries: int = 0           # every delivery record, including wides and no-balls
    start_delivery_id: str = ""
    end_delivery_id: str = ""
    boundary_reason: str = ""     # why it ended: dismissal:<kind>, retirement:<kind>, replacement, innings_end
    quality_status: str = "ok"    # 'warning' when the end could not be explained

    def as_row(self) -> dict[str, Any]:
        return asdict(self)


@dataclass
class InningsExtractionResult:
    """Everything the extractor learns from one innings."""

    observations: list[PartnershipObservation] = field(default_factory=list)
    warnings: list[dict[str, Any]] = field(default_factory=list)
    runs_from_deliveries: int = 0   # all runs.total in the innings (used to check the partnerships)
    wicket_records: int = 0
    delivery_count: int = 0


def make_delivery_id(innings_index: int, over_number: int, ball_position: int) -> str:
    """Stable ID of a delivery: innings, over (0-based, as in Cricsheet), position in the over (1-based)."""
    return f"{innings_index}.{over_number}.{ball_position}"


def is_retirement(wicket_kind: str) -> bool:
    """True for 'retired hurt', 'retired out' and similar kinds."""
    return wicket_kind.lower().startswith("retired")


def delivery_replaces_batter(delivery: dict[str, Any], leaving_batters: set[str]) -> bool:
    """True if this delivery records a replacement that takes one of ``leaving_batters`` off the field."""
    replacement_records = delivery.get("replacements", {}) or {}
    for role_change in replacement_records.get("role", []) or []:
        if role_change.get("role") == "batter" and role_change.get("out") in leaving_batters:
            return True
    for match_change in replacement_records.get("match", []) or []:   # e.g. concussion substitutes
        if match_change.get("out") in leaving_batters:
            return True
    return False

## The innings extractor

The function walks through one innings and keeps the **current pair** as its state. The comments follow the
rules listed at the top of the notebook.

In [ ]:
def extract_innings_partnerships(innings_data: dict[str, Any], player_registry: dict[str, str],
                                 match_id: str, innings_index: int) -> InningsExtractionResult:
    """Extract all partnership observations of one innings.

    ``player_registry`` is ``info.registry.people`` from the match file. It maps the name used in the file
    to the stable Cricsheet person ID, so the same player always gets the same node.
    """
    result = InningsExtractionResult()

    def add_warning(warning_code: str, delivery_id: str, detail: str) -> None:
        result.warnings.append({"match_id": match_id, "innings_index": innings_index,
                                "delivery_id": delivery_id, "code": warning_code, "detail": detail})

    def player_id_of(player_name: str, delivery_id: str) -> str:
        if player_name in player_registry:
            return player_registry[player_name]
        add_warning("player_not_in_registry", delivery_id, player_name)
        return f"unresolved:{player_name}"

    current_partnership: PartnershipObservation | None = None
    current_pair_names: set[str] = set()
    previous_delivery: dict[str, Any] | None = None
    batters_just_out: list[tuple[str, str]] = []   # (name, wicket kind) from the last wicket ball

    def start_partnership(batter_name: str, non_striker_name: str, delivery_id: str) -> PartnershipObservation:
        # Sort the two players by ID so the pair always has the same key.
        first_player, second_player = sorted([(player_id_of(batter_name, delivery_id), batter_name),
                                              (player_id_of(non_striker_name, delivery_id), non_striker_name)])
        return PartnershipObservation(
            match_id=match_id, innings_index=innings_index,
            partnership_seq=len(result.observations) + 1,
            player_1_id=first_player[0], player_1_name=first_player[1],
            player_2_id=second_player[0], player_2_name=second_player[1],
            start_delivery_id=delivery_id)

    def close_partnership(reason: str, status: str = "ok") -> None:
        nonlocal current_partnership, current_pair_names
        current_partnership.boundary_reason = reason
        if status != "ok":
            current_partnership.quality_status = status
        result.observations.append(current_partnership)
        current_partnership, current_pair_names = None, set()

    for over in innings_data.get("overs", []):
        over_number = over.get("over")
        for ball_position, delivery in enumerate(over.get("deliveries", []), start=1):
            delivery_id = make_delivery_id(innings_index, over_number, ball_position)
            pair_names_now = {delivery["batter"], delivery["non_striker"]}
            result.delivery_count += 1

            # A delivery must name two different batters, otherwise no pair can be formed.
            if len(pair_names_now) != 2:
                add_warning("batter_equals_non_striker", delivery_id, delivery["batter"])
                result.runs_from_deliveries += int(delivery["runs"]["total"])
                continue

            # Check the step after a wicket: a batter who was out must not still be at the crease.
            if batters_just_out:
                for out_name, wicket_kind in batters_just_out:
                    if out_name in pair_names_now:
                        add_warning("player_out_still_at_crease", delivery_id, f"{out_name} ({wicket_kind})")
                batters_just_out = []

            # Rule 4: the pair changed but no wicket closed the partnership.
            if current_partnership is not None and pair_names_now != current_pair_names:
                leaving_batters = current_pair_names - pair_names_now
                replacement_recorded = delivery_replaces_batter(delivery, leaving_batters) or (
                    previous_delivery is not None and delivery_replaces_batter(previous_delivery, leaving_batters))
                if replacement_recorded:
                    close_partnership("replacement")
                else:
                    add_warning("unexplained_pair_change", delivery_id,
                                f"{sorted(current_pair_names)} -> {sorted(pair_names_now)}")
                    close_partnership("unexplained_pair_change", "warning")

            if current_partnership is None:
                current_partnership = start_partnership(delivery["batter"], delivery["non_striker"], delivery_id)
                current_pair_names = pair_names_now

            # Rule 2: add runs.total exactly once. Batter runs and extras are kept only for checking.
            delivery_runs = delivery["runs"]
            current_partnership.partnership_runs += int(delivery_runs["total"])
            current_partnership.batter_runs += int(delivery_runs.get("batter", 0))
            current_partnership.extras_runs += int(delivery_runs.get("extras", 0))
            current_partnership.deliveries += 1
            current_partnership.end_delivery_id = delivery_id
            result.runs_from_deliveries += int(delivery_runs["total"])

            # Rule 3: look at EVERY wicket record on this ball (there can be more than one).
            wicket_records = delivery.get("wickets", []) or []
            result.wicket_records += len(wicket_records)
            batters_leaving = []
            for wicket in wicket_records:
                out_name, wicket_kind = wicket.get("player_out", ""), str(wicket.get("kind", "unknown"))
                if out_name in current_pair_names:
                    batters_leaving.append((out_name, wicket_kind))
                else:
                    add_warning("player_out_not_at_crease", delivery_id, f"{out_name} ({wicket_kind})")
            if batters_leaving:
                all_kinds = "+".join(kind for _, kind in batters_leaving)
                reason_type = "retirement" if all(is_retirement(kind) for _, kind in batters_leaving) else "dismissal"
                batters_just_out = batters_leaving
                close_partnership(f"{reason_type}:{all_kinds}")
            previous_delivery = delivery

    # Rule 5: close the last partnership once when the innings ends.
    if current_partnership is not None:
        close_partnership("innings_end")
    return result

## From matches to tables

`extract_match_partnerships` runs the extractor on every **England** innings of one match. Super-over
innings are listed but are not used, because they are not part of the normal match.

In [ ]:
OBSERVATION_COLUMNS = [
    "match_id", "source_file", "match_date", "format", "innings_index", "batting_team",
    "partnership_seq", "player_1_id", "player_1_name", "player_2_id", "player_2_name",
    "partnership_runs", "batter_runs", "extras_runs", "deliveries",
    "start_delivery_id", "end_delivery_id", "boundary_reason", "quality_status",
]
WARNING_COLUMNS = ["match_id", "source_file", "format", "innings_index", "delivery_id", "code", "detail"]


def extract_match_partnerships(match_data: dict[str, Any], match_record: dict[str, Any]):
    """Return (observation rows, innings rows, warning rows) for one qualifying match."""
    player_registry = match_data["info"].get("registry", {}).get("people", {})
    observation_rows, innings_rows, warning_rows = [], [], []
    for innings_index, innings_data in enumerate(match_data.get("innings", []), start=1):
        if innings_data.get("team") != TEAM_NAME:
            continue   # only England's batting innings are studied
        is_super_over = bool(innings_data.get("super_over", False))
        penalty_runs = innings_data.get("penalty_runs", {}) or {}
        innings_row = {
            "match_id": match_record["match_id"], "format": match_record["format"],
            "match_date": match_record["start_date"], "innings_index": innings_index,
            "batting_team": innings_data.get("team"),
            "qualifying": not is_super_over, "super_over": is_super_over,
            "declared": bool(innings_data.get("declared", False)),
            "forfeited": bool(innings_data.get("forfeited", False)),
            "absent_hurt": "|".join(innings_data.get("absent_hurt", []) or []),
            "penalty_runs_pre": int(penalty_runs.get("pre", 0)),
            "penalty_runs_post": int(penalty_runs.get("post", 0)),
            "runs_from_deliveries": 0, "partnership_runs_sum": 0, "wicket_records": 0,
            "delivery_count": 0, "n_partnerships": 0,
        }
        if not is_super_over:
            innings_result = extract_innings_partnerships(innings_data, player_registry,
                                                          match_record["match_id"], innings_index)
            for observation in innings_result.observations:
                observation_row = observation.as_row()
                observation_row.update(format=match_record["format"], match_date=match_record["start_date"],
                                       source_file=match_record["filename"], batting_team=innings_data.get("team"))
                observation_rows.append(observation_row)
            for warning in innings_result.warnings:
                warning.update(format=match_record["format"], source_file=match_record["filename"])
                warning_rows.append(warning)
            innings_row.update(
                runs_from_deliveries=innings_result.runs_from_deliveries,
                partnership_runs_sum=sum(obs.partnership_runs for obs in innings_result.observations),
                wicket_records=innings_result.wicket_records,
                delivery_count=innings_result.delivery_count,
                n_partnerships=len(innings_result.observations))
        innings_rows.append(innings_row)
    return observation_rows, innings_rows, warning_rows


def extract_all_matches(included_matches: pd.DataFrame, match_folder: Path = RAW_MATCH_FOLDER):
    """Run the extractor over every included match. Returns observation, innings and warning tables."""
    all_observations, all_innings, all_warnings = [], [], []
    for match_record in included_matches.to_dict("records"):
        match_data = load_match_file(match_folder / match_record["filename"])
        observation_rows, innings_rows, warning_rows = extract_match_partnerships(match_data, match_record)
        all_observations += observation_rows
        all_innings += innings_rows
        all_warnings += warning_rows
    return (pd.DataFrame(all_observations, columns=OBSERVATION_COLUMNS),
            pd.DataFrame(all_innings),
            pd.DataFrame(all_warnings, columns=WARNING_COLUMNS))